In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
import shap
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from datetime import date
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
import optuna
from optuna.samplers import TPESampler

1 Загрузка и обработка данных


In [5]:
train = pd.read_csv('/Users/andrejbatutin/Whisper/shift_ml_2026_train.csv')
# train = train.sample(frac=1/3, random_state=42)      #  Создаем уменьшенную случайную выборку

/var/folders/hs/82_18tb93256yd1jg5n3fmwc0000gn/T/ipykernel_34964/2309189368.py:1: DtypeWarning: Columns (49) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('/Users/andrejbatutin/Whisper/shift_ml_2026_train.csv')


In [6]:
def prepare(data):
    limit = len(data) * 0.7
    data = data.dropna(thresh=limit, axis=1)              # удалить колонки, где более 30% пропусков - оставила те, где хотя бы 70% данных (0.7) заполнены
    data = data.fillna(data.median(numeric_only=True))   # пропуски в числовых признаках заполняем медианой
    data['профессия_заемщика'] = data['профессия_заемщика'].fillna('unknown')
    data['стаж'] = data['стаж'].fillna('unknown')         # остались пропуски в двух кат признаках, заполняем 'unknown'
    x_train = data.drop(columns=['итоговый_статус_займа', 'id']) # на основе чего предсказываем
    y_train = data['итоговый_статус_займа']                      # то, что предсказываем
    return x_train , y_train

x_train,y_train = prepare(train)

2 Созданий фичей

In [7]:
def features(data):
    # 1. Оригинальные фичи (исправленные)
    data['дата_первого_займа'] = pd.to_datetime(data['дата_первого_займа'])
    current_date = pd.to_datetime('today').normalize()
    data['credit_history_age_months'] = (current_date - data['дата_первого_займа']).dt.days / 30.44

    data['has_active_delinquency'] = ((data['кол-во_счетов_с_акт_просрочкой_1_мес'] > 0) |
                                      (data['кол-во_счетов_с_акт_просрочкой_3_мес_за_2_года'] > 0)).astype(int)

    data['Debt_burden'] = (data['аннуитет'] * 12) / (data['годовой_доход'] + 1e-6)
    data['credit_income_ratio'] = data['сумма_займа'] / (data['годовой_доход'] + 1e-6)
    data['annuitet_to_the_card_limit'] = data['аннуитет'] / (data['общий_лимит_по_возоб_счету'] + 1e-6)

    # ИСПРАВЛЕНО: добавлены скобки
    data['percentage_of_accounts_overdue'] = (
        data['кол-во_счетов_с_просрочкой_4_мес'] +
        data['кол-во_счетов_с_акт_просрочкой_1_мес'] +
        data['кол-во_счетов_с_акт_просрочкой_3_мес_за_2_года']
    ) / (data['кол-во_счетов'] + 1e-6)

    data['overdue'] = (data['кол-во_просрочек_за_два_года'] > 0).astype(int)
    data['is_bankrupt'] = (data['кол-во_публ_банкротств'] > 0).astype(int)
    data['Utilization_Ratio'] = data['общая_сумма_на_счетах'] / (data['общий_лимит_по_возоб_счету'] + 1e-6)
    data['ratio_of_collection_payments'] = data['сумма_выплат_по_просрочкам'] / (data['общая_сумма_взысканий'] + 1e-6)
    data['debt_to_loan_amount'] = data['сумма_задолженности'] / (data['сумма_займа'] + 1e-6)

    # Группировка категорий
    threshold = 100
    popular_professions = data['профессия_заемщика'].value_counts()[
        data['профессия_заемщика'].value_counts() >= threshold
    ].index
    data['профессия_заемщика_grouped'] = np.where(
        data['профессия_заемщика'].isin(popular_professions),
        data['профессия_заемщика'],
        'Other'
    )

    top_regions = data['регион'].value_counts().nlargest(20).index
    data['регион_grouped'] = np.where(
        data['регион'].isin(top_regions),
        data['регион'],
        'Other'
    )

    # 2. НОВЫЕ МОЩНЫЕ ФИЧИ

    # 2.1 Агрегированные финансовые метрики
    data['total_debt_to_income'] = (data['общая_сумма_на_счетах'] + data['сумма_займа']) / (data['годовой_доход'] + 1e-6)
    data['revolving_accounts_ratio'] = data['кол-во_возобновляемых_счетов'] / (data['кол-во_счетов'] + 1e-6)
    data['avg_balance_per_account'] = data['общая_сумма_на_счетах'] / (data['кол-во_счетов'] + 1e-6)
    data['avg_credit_limit_per_account'] = data['общий_лимит_по_возоб_счету'] / (data['кол-во_счетов'] + 1e-6)

    # 2.2 Временные паттерны и активность
    data['account_density_per_month'] = data['кол-во_счетов'] / (data['credit_history_age_months'] + 1)
    data['recent_accounts_ratio'] = data['кол-во_счетов_за_посл_год'] / (data['кол-во_счетов'] + 1)
    data['months_since_last_activity'] = data[
        ['кол-во_месяцев_с_последнего_счета',
         'кол-во_месяцев_с_последней_карты',
         'кол-во_мес_с_последней_заявки']
    ].min(axis=1)

    # 2.3 Рисковые индикаторы
    data['delinquency_score'] = (
        data['кол-во_счетов_с_акт_просрочкой_1_мес'] * 1 +
        data['кол-во_счетов_с_акт_просрочкой_3_мес_за_2_года'] * 3 +
        data['кол-во_счетов_с_просрочкой_4_мес'] * 5
    )

    data['overleveraged'] = (
        (data['Utilization_Ratio'] > 0.8) |
        (data['credit_income_ratio'] > 3) |
        (data['Debt_burden'] > 0.5)
    ).astype(int)

    # 2.4 Коэффициенты использования разных типов кредитов
    data['card_utilization_rate'] = data['соотношение_баланса_к_лимиту_по_картам'].fillna(0)
    data['revolving_utilization_rate'] = data['коэфф_загрузки_возобновляемого_счета'].fillna(0)

    # 2.5 Плотность просрочек
    data['delinquency_density'] = (
        data['кол-во_просрочек_за_два_года'] /
        (data['credit_history_age_months'] / 24 + 1e-6)
    )

    # 2.6 Интеракции важных признаков
    data['rating_income_interaction'] = data['рейтинг_OCR'] * np.log1p(data['годовой_доход'])
    data['loan_amount_rate_interaction'] = data['сумма_займа'] * data['процентная_ставка']

    # 2.7 Сегментация по стажу (binning)
    data['стаж_numeric'] = data['стаж'].map({
        'unknown': -1,
        '< 1 года': 0,
        '1 год': 1,
        '2 года': 2,
        '3 года': 3,
        '4 года': 4,
        '5 лет': 5,
        '6 лет': 6,
        '7 лет': 7,
        '8 лет': 8,
        '9 лет': 9,
        '10+ лет': 10
    })

    data['experience_credit_interaction'] = data['стаж_numeric'] * data['credit_history_age_months']

    # 2.8 Отношение платежей к доходам
    data['monthly_payment_to_income'] = data['аннуитет'] / (data['годовой_доход'] / 12 + 1e-6)

    # 2.9 Качество кредитной истории
    data['good_accounts_ratio'] = data['кол-во_счетов_без_нарушений'] / (data['кол-во_счетов'] + 1e-6)
    data['high_limit_accounts_ratio'] = data['процент_счетов_прев_75_лимита'] / 100

    # 2.10 Временные индикаторы
    data['is_recent_borrower'] = (data['credit_history_age_months'] < 12).astype(int)
    data['is_established_borrower'] = (data['credit_history_age_months'] > 60).astype(int)

    # 2.11 Кластеризация по финансовому поведению
    data['financial_stress_score'] = (
        data['percentage_of_accounts_overdue'] * 0.3 +
        data['Utilization_Ratio'] * 0.3 +
        data['Debt_burden'] * 0.2 +
        data['delinquency_density'] * 0.2
    )

    # 2.12 Логарифмические трансформации для skewed признаков
    data['log_годовой_доход'] = np.log1p(data['годовой_доход'])
    data['log_сумма_займа'] = np.log1p(data['сумма_займа'])
    data['log_общая_сумма_на_счетах'] = np.log1p(data['общая_сумма_на_счетах'].clip(lower=0))

    # 2.13 Полиномиальные фичи
    data['годовой_доход_squared'] = data['годовой_доход'] ** 2
    data['credit_history_age_squared'] = data['credit_history_age_months'] ** 2

    # 2.14 Отношения лимитов
    data['total_limit_to_income'] = data['общий_лимит_по_возоб_счету'] / (data['годовой_доход'] + 1e-6)
    data['available_credit_ratio'] = (data['общий_лимит_по_возоб_счету'] - data['общая_сумма_на_счетах']) / (
        data['общий_лимит_по_возоб_счету'] + 1e-6)

    # 2.15 Частотное кодирование для категориальных признаков
    for col in ['профессия_заемщика_grouped', 'регион_grouped', 'цель_займа']:
        freq = data[col].value_counts(normalize=True)
        data[f'{col}_freq'] = data[col].map(freq).fillna(0)

    # 2.16 Специальные фичи для кредитных карт
    data['active_cards_ratio'] = data['кол-во_активных_карт'] / (data['кол-во_карт'] + 1e-6)
    data['cards_without_delinquency_ratio'] = data['кол-во_карт_без_нарушений'] / (data['кол-во_карт'] + 1e-6)

    # 2.17 Индикаторы разнообразия кредитного портфеля
    credit_types = data[['кол-во_карт', 'кол-во_аннуитетных_счетов', 'кол-во_возобновляемых_счетов']].values
    data['credit_portfolio_diversity'] = credit_types.std(axis=1) / (credit_types.mean(axis=1) + 1e-6)

    # 2.18 Возраст самого старого счета
    data['oldest_account_age'] = data[
        ['кол-во_месяцев_с_первого_аннуитетного_счета',
         'кол-во_месяцев_с_первого_возобновляемого_счета']
    ].max(axis=1)

    # 2.19 Комбинации категориальных признаков
    data['profession_region_combo'] = data['профессия_заемщика_grouped'] + '_' + data['регион_grouped']

    # 2.20 Индикаторы для особых случаев
    data['has_mortgage'] = (data['кол-во_ипотек'] > 0).astype(int)
    data['has_car_loan'] = (data['кол-во_залогов'] > 0).astype(int)

    # 2.21 Нормализованные баллы
    data['normalized_delinquency'] = (
        data['delinquency_score'] - data['delinquency_score'].mean()
    ) / (data['delinquency_score'].std() + 1e-6)

    # 2.22 Квантильные признаки
    data['income_quantile'] = pd.qcut(
        data['годовой_доход'],
        q=5,
        labels=False,
        duplicates='drop'
    )

    # 2.23 Взаимодействие типа займа с другими признаками
    loan_type_mapping = {
        'потребительский': 1,
        'автокредит': 2,
        'ипотека': 3,
        'кредитная карта': 4
    }
    data['loan_type_numeric'] = data['тип_займа'].map(loan_type_mapping).fillna(1)
    data['loan_type_income_interaction'] = data['loan_type_numeric'] * data['log_годовой_доход']

    # 3. Обработка бесконечностей и пропусков
    numeric_cols = data.select_dtypes(include=[np.number]).columns

    # Замена бесконечностей
    data = data.replace([np.inf, -np.inf], np.nan)

    # Заполнение пропусков медианой
    for col in numeric_cols:
        if data[col].isnull().any():
            data[col] = data[col].fillna(data[col].median())

    # 4. Удаление дубликатов и константных признаков (в конце)
    # (это можно сделать после применения функции)

    return data

In [8]:
x_train = features(x_train)

/var/folders/hs/82_18tb93256yd1jg5n3fmwc0000gn/T/ipykernel_34964/3150418553.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['дата_первого_займа'] = pd.to_datetime(data['дата_первого_займа'])
/var/folders/hs/82_18tb93256yd1jg5n3fmwc0000gn/T/ipykernel_34964/3150418553.py:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['monthly_payment_to_income'] = data['аннуитет'] / (data['годовой_доход'] / 12 + 1e-6)
/var/folders/hs/82_18tb93256yd1jg5n3fmwc0000gn/T/ipykernel_34964/3150418553.py:111: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whic

3 Важность SHAP

In [9]:
cat_features = x_train.select_dtypes(include=['object']).columns.to_list() # Указываем для модели, какие колонки категориальные
model = CatBoostClassifier(       # CatBoostClassifier - библиотека машинного обучения, работает намного лучше и быстрее логистической регрессии, особенно на «грязных» данных с текстом
            iterations=1000,              # количество деревьев. Большее значение может повысить точность, но увеличивает время обучения.
            learning_rate=0.05,            # темп обучения (0.01–0.1) Меньшее значение позволяет модели учиться более плавно и уменьшает риск переобучения
            depth=6,                      # глубина деревьев регулирует сложность модели. Глубокие деревья могут лучше выявлять сложные зависимости, но также рискуют переобучиться.
            cat_features=cat_features,    # набор данных
            verbose=300,                  # показывать прогресс каждые 100 шагов
            random_state = 42,
            thread_count=-1,
            eval_metric = 'AUC')
model.fit(x_train, y_train)

explainer = shap.TreeExplainer(model)
explanation = explainer(x_train)
# shap.plots.bar(explanation)
mean_shap_values = np.abs(explanation.values).mean(axis=0)   # explanation.values содержит SHAP значения для каждого объекта и каждого признака,усреднить их по всем объектам (ось 0) и взять абсолютное значение
feature_importance_df = pd.DataFrame({                       # Создание DataFrame для удобства анализа
    'Feature': x_train.columns,                              # Используем имена признаков из вашего DataFrame
    'SHAP_Importance': mean_shap_values})
feature_importance_df = feature_importance_df.sort_values(by='SHAP_Importance', ascending=False)   # Сортировка по важности
print("Топ-20 признаков по важности SHAP:")
print(feature_importance_df.head(20))

zero_importance_features = feature_importance_df[feature_importance_df['SHAP_Importance'] == 0]['Feature'].tolist()   # Список признаков с нулевой важностью
print(f"\nПризнаки с нулевой важностью: {zero_importance_features}")

0:	total: 461ms	remaining: 7m 41s
300:	total: 2m 12s	remaining: 5m 8s
600:	total: 4m 26s	remaining: 2m 57s
900:	total: 6m 35s	remaining: 43.4s
999:	total: 7m 19s	remaining: 0us
Топ-20 признаков по важности SHAP:
                                   Feature  SHAP_Importance
1                               срок_займа         0.245481
5                               допрейтинг         0.184263
18                                     пдн         0.127435
4                                  рейтинг         0.104610
45        кол-во_открытых_счетов_за_2_года         0.097538
6                       профессия_заемщика         0.092900
93              профессия_заемщика_grouped         0.088681
8                          владение_жильем         0.087996
91            ratio_of_collection_payments         0.085751
17                                  регион         0.071110
85                     credit_income_ratio         0.060699
2                        процентная_ставка         0.059856
35      

In [10]:
X_train_clean = x_train.drop(columns=zero_importance_features)   # Создание нового X_train без нулевых признаков

In [11]:
cat_features_clean = X_train_clean.select_dtypes(include=['object']).columns.to_list() # Указываем для модели, какие колонки категориальные

4 Optuna — мощный и гибкий фреймворк для автоматизированного подбора гиперпараметров

In [12]:
def catboost_objective(trial):      # Это основная функция, которую Optuna будет вызывать для каждого эксперимента (пробного запуска, trial). Она определяет, какие параметры тестировать и как оценивать их качество.
    global all_fe
    all_fe = pd.DataFrame()

    params = {
        'iterations': trial.suggest_int('iterations', 500, 5000),    # suggest_int/float говорят Optuna выбрать случайное значение из указанного диапазона
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'depth': trial.suggest_int('depth', 3, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0, 5),     #  (L2-регуляризация)
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_strength': trial.suggest_float('random_strength', 0, 2),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 5),
        'auto_class_weights' : trial.suggest_categorical('auto_class_weights', ['Balanced', None]),   # выбирает одно значение из списка


    }    # Здесь определяются диапазоны гиперпараметров, которые Optuna будет исследовать.


    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # стратифицированная кросс-валидации на 3 фолдов. Это гарантирует, что в каждом из 3 экспериментов сохраняется пропорциональный баланс классов (0 и 1), что важно для несбалансированных данных.

    auc_scores = []
    for train_idx, val_idx in cv.split(X_train_clean, y_train):
        x_train_fold, x_val_fold = X_train_clean.iloc[train_idx], X_train_clean.iloc[val_idx]   # Данные делятся на обучающую и валидационную части
        y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]   # итерируется по различным парам тренировочных и валидационных индексов (train_idx, val_idx), которые выделяем из всего набора данных (x_train, y_train):


        model_optuna = CatBoostClassifier(**params,                  # Инициализируется CatBoostClassifier с параметрами, предложенными для Optuna для текущего trial.
                                   random_seed=42,
                                   loss_function='Logloss',
                                   cat_features=cat_features_clean,
                                   thread_count=-1,
                                   eval_metric = 'AUC'
                                  )
        model_optuna.fit(x_train_fold, y_train_fold, eval_set=(x_val_fold, y_val_fold), early_stopping_rounds=150, verbose=False) # Модель обучается, используя eval_set (валидационный набор) для отслеживания метрики и early_stopping_rounds (остановка, если метрика не улучшается 150 раундов).
        y_pred_proba = model_optuna.predict_proba(x_val_fold)[:, 1]
        auc = roc_auc_score(y_val_fold, y_pred_proba)     # Рассчитывается roc_auc_score на валидационной выборке
        auc_scores.append(auc)
        few = pd.DataFrame({'w':model_optuna.feature_importances_,'name':X_train_clean.columns}).sort_values('w',ascending=False).reset_index(drop=True) # Рассчитывается и собирается важность признаков (feature_importances_) в DataFrame all_fe.
        few['index'] = 264 - few.index
        all_fe = pd.concat([all_fe]+[few])

    print(f'Trial {trial.number}: {auc_scores} , mean : {np.mean(auc_scores)}')
    return np.mean(auc_scores)      # Функция возвращает средний ROC AUC по всем 5 фолдам. Optuna использует это значение для оценки качества набора гиперпараметров в данном trial и принимает решение, какие параметры попробовать в следующем.

# Эта часть управляет процессом оптимизации

# direction='maximize' говорит Optuna, что цель — найти параметры, которые максимизируют возвращаемое значение (средний ROC AUC).
# sampler=TPESampler(...) определяет алгоритм, который Optuna использует для выбора "умных" параметров для следующего trial на основе предыдущих результатов (TPESampler — это популярный и эффективный алгоритм)
catboost_study = optuna.create_study(direction='maximize', sampler=TPESampler(n_startup_trials=25, seed=42, multivariate=True))  # Создает объект исследования Optuna
catboost_study.optimize(catboost_objective, n_trials=50)  #Запускает процесс оптимизации. Он вызывает функцию catboost_objective указанное количество раз (n_trials=50).


best_params = catboost_study.best_params   # После завершения всех пробных запусков, здесь будут храниться параметры, которые дали наивысший средний ROC AUC
print("Лучшие параметры:", best_params)
all_fe.groupby("name")['w'].agg(['mean','sum','count']).sort_values('mean',ascending=False)   # анализирует собранную важность признаков

/var/folders/hs/82_18tb93256yd1jg5n3fmwc0000gn/T/ipykernel_34964/513364175.py:49: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  catboost_study = optuna.create_study(direction='maximize', sampler=TPESampler(n_startup_trials=25, seed=42, multivariate=True))  # Создает объект исследования Optuna
[I 2026-02-05 08:04:40,728] A new study created in memory with name: no-name-565e9769-9ebd-420d-9d30-54e0c7189881
[I 2026-02-05 08:19:12,715] Trial 0 finished with value: 0.759864837376156 and parameters: {'iterations': 2185, 'learning_rate': 0.28570714885887566, 'depth': 8, 'l2_leaf_reg': 2.993292420985183, 'border_count': 66, 'random_strength': 0.3119890406724053, 'bagging_temperature': 0.2904180608409973, 'auto_class_weights': 'Balanced'}. Best is trial 0 with value: 0.759864837376156.


Trial 0: [0.7592162632503001, 0.7594063995741277, 0.7605228111820086, 0.7596655686149125, 0.7605131442594308] , mean : 0.759864837376156


[W 2026-02-05 12:15:04,799] Trial 1 failed with parameters: {'iterations': 3687, 'learning_rate': 0.01596950334578271, 'depth': 10, 'l2_leaf_reg': 4.162213204002109, 'border_count': 79, 'random_strength': 0.36364993441420124, 'bagging_temperature': 0.9170225492671691, 'auto_class_weights': None} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "/Users/andrejbatutin/Whisper/.venv/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/hs/82_18tb93256yd1jg5n3fmwc0000gn/T/ipykernel_34964/513364175.py", line 34, in catboost_objective
    model_optuna.fit(x_train_fold, y_train_fold, eval_set=(x_val_fold, y_val_fold), early_stopping_rounds=150, verbose=False) # Модель обучается, используя eval_set (валидационный набор) для отслеживания метрики и early_stopping_rounds (остановка, если метрика не улучшается 150 раундов).
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

{'iterations': 2185,
 'learning_rate': 0.28570714885887566,
 'depth': 8,
 'l2_leaf_reg': 2.993292420985183,
 'border_count': 66,
 'random_strength': 0.3119890406724053,
 'bagging_temperature': 0.2904180608409973,
 'auto_class_weights': 'Balanced'}

5 Обучаем финальную модель

In [18]:
import json

# Пример: если hyperparams - это строка
hyperparams_str = '{"random_strength":0.3119890407,"border_count":66,"iterations":2185,"bagging_temperature":0.2904180608,"auto_class_weights":"Balanced","l2_leaf_reg":2.993292421,"depth":8,"learning_rate":0.2857071489}'

# Преобразуем строку в словарь
hyperparams = json.loads(hyperparams_str)

# Преобразуем целочисленные значения
int_params = ['iterations', 'depth', 'border_count']
for param in int_params:
    if param in hyperparams:
        hyperparams[param] = int(hyperparams[param])

print("Преобразованные параметры:", hyperparams)
print("Тип iterations:", type(hyperparams['iterations']))


Преобразованные параметры: {'random_strength': 0.3119890407, 'border_count': 66, 'iterations': 2185, 'bagging_temperature': 0.2904180608, 'auto_class_weights': 'Balanced', 'l2_leaf_reg': 2.993292421, 'depth': 8, 'learning_rate': 0.2857071489}
Тип iterations: <class 'int'>


In [19]:
# Инициализируем финальную модель с лучшими параметрами
model_final = CatBoostClassifier(
    **hyperparams,
    random_seed=42,
    loss_function='Logloss',
    cat_features=cat_features_clean,
    thread_count=-1,
    eval_metric='AUC',
    verbose=0
)

# Обучаем модель на ВСЕХ тренировочных данных
print("Начинаем обучение финальной модели на полном тренировочном наборе данных...")
model_final.fit(X_train_clean, y_train)
print("Обучение завершено.")

Начинаем обучение финальной модели на полном тренировочном наборе данных...
Обучение завершено.


6 Загружаю тестовую выборку

In [23]:
model_final.save_model('model_final_shapoptuna0502.cbm')

In [ ]:
test = pd.read_csv('/Users/andrejbatutin/Whisper/shift_ml_2026_test.csv')
client_ids = test['id']  # Сохраняем ID отдельно, чтобы потом соединить с результатом
limit = len(test) * 0.7
test = test.dropna(thresh=limit, axis=1)              # удалить колонки, где более 30% пропусков - оставила те, где хотя бы 70% данных (0.7) заполнены
test = test.fillna(x_train.median(numeric_only=True))   # пропуски в числовых признаках заполняем медианой
test['профессия_заемщика'] = test['профессия_заемщика'].fillna('unknown')
test['стаж'] = test['стаж'].fillna('unknown')         # остались пропуски в двух кат признаках, заполняем 'unknown'
x_test = test.drop(columns=['id'])
x_test = features(x_test)
x_test_clean = x_test.drop(columns=zero_importance_features)   # Создание нового x_test без признаков нулевой важности
# Получаем вероятности
result_proba = model_final.predict_proba(x_test_clean)[:, 1]
if len(client_ids) == len(result_proba):    # Убедиться, что количество ID и количество вероятностей совпадают
    results_df = pd.DataFrame({
    'ID': client_ids,
    'Proba': result_proba
})
    results_df.to_csv('submission0502.csv', index=False)
else:
    print("Ошибка: Количество ID клиентов не совпадает с количеством предсказаний.")

/var/folders/hs/82_18tb93256yd1jg5n3fmwc0000gn/T/ipykernel_34964/3150418553.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['дата_первого_займа'] = pd.to_datetime(data['дата_первого_займа'])
/var/folders/hs/82_18tb93256yd1jg5n3fmwc0000gn/T/ipykernel_34964/3150418553.py:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['monthly_payment_to_income'] = data['аннуитет'] / (data['годовой_доход'] / 12 + 1e-6)
/var/folders/hs/82_18tb93256yd1jg5n3fmwc0000gn/T/ipykernel_34964/3150418553.py:111: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whic